In [0]:
dbutils.widgets.text("entity_name", "products")
entity_name = dbutils.widgets.get("entity_name")

In [0]:
%run ../07_Common/00_setup

In [0]:
%run ../07_Common/03_silver/Utils_Control_Config_Silver

In [0]:
%run ../07_Common/03_silver/Utils_Flatten

In [0]:
%run ../07_Common/03_silver/Utils_Deduplicacion_Silver

In [0]:

print(f" [{entity_name}] Iniciando transformación a Silver...")

silver_config = get_silver_config(entity_name)

source_table       = silver_config["source_table"]
target_table       = silver_config["target_table"]
primary_key        = silver_config["primary_key"]
dedup_order_column = silver_config["dedup_order_column"]

print(f"   Origen: {source_table} → Destino: {target_table}")
print(f"   Llave: {primary_key} | Orden de dedup: {dedup_order_column}")

In [0]:
df_bronze = spark.table(source_table)
print(f"  Registros leídos desde Bronze: {df_bronze.count()}")

In [0]:
df_flat = flatten_structs(df_bronze)
df_flat = standardize_column_names(df_flat)
print(f" Columnas finales: {df_flat.columns}")

In [0]:
df_dedup = deduplicate(df_flat, key_columns=[primary_key], order_column=dedup_order_column)
print(f"  Registros tras deduplicación: {df_dedup.count()} (de {df_flat.count()} antes)")


In [0]:
try:
    (
        df_dedup.write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )
    estado_final = "success"
    print(f"   Escritura exitosa en {target_table}")
except Exception as e:
    estado_final = "failed"
    print(f"  ERROR al escribir en Silver: {e}")